# Modelado de Toomer 

1. *Núcleos puntuales* Representan toda la masa de cada galaxia, es decir las interacciones gravitacionales se dan con y entre los núcleos. 

2. *Estrellas como partículas de prueba* i.e. la masa de la estrellas es $m_{k} =0$ y esto quiere decir que interactúan entre ellas. 

In [3]:
import numpy as np 
import matplotlib.pyplot as plt 
from numba import jit 
from typing import List, Dict, Any

In [4]:
class N_system: 
    def __init__(self, particles: List[Dict[str, Any]]):
        ''' Clase general para un sistema de N partículas, 
         donde cada particula es un diccionario con sus propiedades: 
         -"mass" : masa partícula 
         - "position": posición inicial (array 3D)
         - "velocity": velocidad inicial (array 3D)
        '''
        self.n = len(particles)
        self.positions = np.array([p['position'] for p in particles])
        self.velocities = np.array([p['velocity'] for p in particles])
        self.masses = np.array([p['mass'] for p in particles])
        self.mass = np.sum(self.masses)

        v_prom = np.mean(np.linalg.norm(self.velocities, axis=1))
        self.t_relax = self.n /np.log(self.n) * (v_prom**3) / \
            (self.mass * 1.0)  # Tiempo de relajación, usando G=1 para simplicidad
        
        ## indices de cuerpos de masa 
        self.mass_indices = np.where(self.masses > 0)[0]

    def evolucionar(self, dt:float, T:float)-> np.ndarray[float,3]:
        ''' Integra el sistema por el metodo de leapfrog durante 
        un tiempo T con paso dt. 
        
        params: 
        - dt: paso de integración
        - T: tiempo total de integración
        
        returns:
        - trayectories: array de forma (n, num_steps, 3)
        con las posiciones de cada partícula en cada paso'''
        
        def acel(pos: np.ndarray[float,3], 
                        masses: np.ndarray[float],
                        m_inidices: np.ndarray[int],
                        n_particles: int) -> np.ndarray[float,3]:
            ''' Calcula la aceleración sobre cada partícula dada su posición actual. 
            Usa la ley de gravitación universal con G=1 para simplicidad.
             
              params:
              - pos: array de forma (n, 3) con las posiciones actuales de las partículas
              - masses: array de forma (n,) con las masas de las partículas
              - m_inidices: array de índices de las partículas con masa positiva'''

            acc = np.zeros_like(n_particles,pos)
            for i in range(self.n):
                for j in m_inidices:
                    if i == j:
                        continue
                    r_vec = pos[i] - pos[j]
                    r_mag = np.linalg.norm(r_vec) + 1e-10  # Evitar división por cero
                    acc[i] -= self.masses[j] * r_vec / r_mag**2
            return acc
        num_steps = int(T / dt)
        trajectories = np.zeros((self.n, num_steps, 3))

        trajectories[:, 0, :] = self.positions

        # Leapfrog integration
        acc = acel(self.positions, self.masses, self.mass_indices, self.n)
        self.velocities += 0.5 * acc * dt  # Kick inicial
        
        for step in range(1,num_steps):
            self.positions += self.velocities * dt  # Drift
            trajectories[:, step, :] = self.positions
            acc = acel(self.positions, self.masses, self.mass_indices, self.n)  # Recalcular aceleración
            self.velocities += acc * dt  # Kick
        return trajectories
        

In [5]:
mi_primer_sistema = N_system([
    {'mass': 1.0, 'position': np.array([0.0,    0.0, 0.0]), 'velocity': np.array([0.0, 0.0, 0.0])},
    {'mass': 2.0, 'position': np.array([1.0,   0.0, 0.0]), 'velocity': np.array([0.0, 1.0, 0.0])},
    {'mass': 3.0, 'position': np.array([0.0,    1.0, 0.0]), 'velocity': np.array([1.0, 0.0, 0.0])}
])




In [6]:
mi_primer_sistema.evolucionar(0.01, 1.0)

TypeError: Cannot construct a dtype from an array